# RAFTcorr â€” GPU-Accelerated Digital Image Correlation

This notebook runs the full **RAFTcorr** application (Flask backend + React frontend) on Google Colab's free T4 GPU.

**Instructions:** Click **Runtime â†’ Run all** (or `Ctrl+F9`), then click the tunnel URL printed in the last cell to open the GUI.

In [ ]:
# Cell 1: Verify GPU is available
!nvidia-smi

In [ ]:
# Cell 2: Clone repo and install Python dependencies
# (torch, numpy, scipy are pre-installed on Colab)
import os

REPO_DIR = "/content/RAFTcorr"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/zachtong/RAFTcorr.git "$REPO_DIR"
else:
    print(f"Repo already cloned at {REPO_DIR}, pulling latest...")
    !cd "$REPO_DIR" && git pull

!pip install -q flask flask-cors flask-socketio pillow tifffile

In [ ]:
# Cell 3: Install Node.js 20 and build React frontend
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - && \
    sudo apt-get install -y nodejs > /dev/null 2>&1
!node --version && npm --version

!cd /content/RAFTcorr/frontend && npm install --legacy-peer-deps && npm run build

## Google Drive (Optional)

Mount your Google Drive to load images from and save results to your Drive.
Skip this cell if you prefer to upload images directly through the GUI.

In [ ]:
# Cell 4: Mount Google Drive (optional â€” skip if not needed)
from google.colab import drive
drive.mount("/content/drive")

# Tip: your Drive files are at /content/drive/MyDrive/
# You can browse them in the Files panel on the left.

In [ ]:
# Cell 5: Start Flask server + cloudflared tunnel
import subprocess, threading, time, re
from IPython.display import display, HTML

# --- 1. Start Flask server in the background ---
flask_proc = subprocess.Popen(
    ["python", "run_prod.py", "--host", "0.0.0.0", "--port", "5000", "--no-browser"],
    cwd="/content/RAFTcorr",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print("Flask server starting...")
time.sleep(3)  # let Flask bind the port

# --- 2. Download cloudflared (free tunnel, no account needed) ---
CF_BIN = "/content/cloudflared"
if not os.path.isfile(CF_BIN):
    !wget -qO "$CF_BIN" \
        https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x "$CF_BIN"

# --- 3. Start cloudflared tunnel ---
tunnel_proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", "http://localhost:5000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# --- 4. Capture the tunnel URL from stderr ---
tunnel_url = None

def _read_tunnel_url():
    global tunnel_url
    for line in tunnel_proc.stderr:
        decoded = line.decode("utf-8", errors="replace")
        match = re.search(r"(https://[a-z0-9-]+\.trycloudflare\.com)", decoded)
        if match:
            tunnel_url = match.group(1)
            return

reader = threading.Thread(target=_read_tunnel_url, daemon=True)
reader.start()
reader.join(timeout=30)

if tunnel_url:
    display(HTML(
        f'<h2>RAFTcorr is ready!</h2>'
        f'<p>Click to open: <a href="{tunnel_url}" target="_blank">{tunnel_url}</a></p>'
    ))
else:
    print("ERROR: Could not obtain tunnel URL within 30 seconds.")
    print("Check that GPU runtime is enabled (Runtime â†’ Change runtime type â†’ T4 GPU).")

## Usage

1. **Click the tunnel URL** above to open the RAFTcorr GUI in a new tab.
2. **Load images** â€” upload reference and deformed image sequences.
   - If you mounted Google Drive, navigate to `/content/drive/MyDrive/...` to find your files.
3. **Set ROI** â€” draw a region of interest on the reference image.
4. **Process** â€” run RAFT-DIC displacement tracking (GPU-accelerated).
5. **Post-process** â€” view displacement/strain fields, export results.
   - Exported files are saved in the server's working directory.
   - To save to Drive: copy results to `/content/drive/MyDrive/`.

**Tip:** The Colab session will disconnect after ~90 minutes of inactivity. Re-run this notebook to restart.